# Cross-Dataset Registration Viewer

Registers one image from **HuashanMyo** against one from **MyosegmenTUM**
(or any two stacks within either dataset) and lets you inspect the result
slice by slice using a **two-colour overlay**:

| Colour | Meaning |
|---|---|
| **Red** | Fixed image only (no coverage from moving) |
| **Cyan** (green+blue) | Moving image only (no coverage from fixed) |
| **Grey** | Overlap — both images present |

Because the two datasets may have different FOV, the images may only overlap
on a subset of slices.  Slices outside the overlap region will appear
pure red or pure cyan.

In [ ]:
import pathlib, re
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox, Button, HTML
from IPython.display import display

EVAL_DIR = pathlib.Path('.')

In [ ]:
# ── Discover stacks in each dataset ──────────────────────────────────────────

def discover_huashanmyo(root):
    stacks = {}
    for w in sorted((root / 'Water').glob('THIGH_*_0001.nii.gz')):
        stem = w.name.replace('_0001.nii.gz', '')
        stacks[f'HuashanMyo / {stem}'] = w
    return stacks


def discover_myosegmentum(root):
    stacks = {}
    for w in sorted(root.glob('*/ImageData/*_WATER/*_WATER_stack*.nii')):
        m = re.match(r'(.+?)_WATER_stack(\d+)\.nii$', w.name)
        if m:
            label = f'MyosegmenTUM / {m.group(1)}_stack{m.group(2)}'
            stacks[label] = w
    return stacks


ALL_STACKS = {}
hm_root  = EVAL_DIR / 'HuashanMyo'
myo_root = EVAL_DIR / 'myosegmenTUM'

if hm_root.exists():
    ALL_STACKS.update(discover_huashanmyo(hm_root))
if myo_root.exists():
    ALL_STACKS.update(discover_myosegmentum(myo_root))

print(f'{len(ALL_STACKS)} stacks available')
for k in list(ALL_STACKS)[:4]:
    print(f'  {k}')

In [ ]:
# ── Registration helper ───────────────────────────────────────────────────────

def register(fixed_path, moving_path, transform='affine'):
    """Register moving onto fixed.  Returns (fixed_arr, registered_arr, mask_arr).

    mask_arr is 1 where the registered image has valid coverage, 0 elsewhere.
    Both output arrays are normalised to [0, 1] and share fixed image's shape.
    """
    fixed  = sitk.ReadImage(str(fixed_path),  sitk.sitkFloat32)
    moving = sitk.ReadImage(str(moving_path), sitk.sitkFloat32)

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.2)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsGradientDescent(
        learningRate=1.0, numberOfIterations=200,
        convergenceMinimumValue=1e-6, convergenceWindowSize=10,
    )
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    tx_type = sitk.AffineTransform(3) if transform == 'affine' else sitk.Euler3DTransform()
    initial = sitk.CenteredTransformInitializer(
        fixed, moving, tx_type,
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )
    reg.SetInitialTransform(initial, inPlace=False)

    print('Registering...', end=' ', flush=True)
    final_tx = reg.Execute(fixed, moving)
    print(f'done  (metric={reg.GetMetricValue():.4f})')

    # Resample moving into fixed space; use -1 as out-of-bounds sentinel
    registered = sitk.Resample(
        moving, fixed, final_tx,
        sitk.sitkLinear, -1.0, moving.GetPixelID(),
    )

    # Build coverage mask: 1 where the moving image had valid data
    ones = sitk.Image(moving.GetSize(), sitk.sitkFloat32) + 1
    ones.CopyInformation(moving)
    mask = sitk.Resample(
        ones, fixed, final_tx,
        sitk.sitkLinear, 0.0, sitk.sitkFloat32,
    )

    def norm(img):
        a = sitk.GetArrayFromImage(img).astype(np.float32)
        valid = a[a >= 0]
        if valid.size == 0:
            return a
        lo, hi = np.percentile(valid, 1), np.percentile(valid, 99)
        return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)

    fixed_arr = norm(fixed)
    reg_arr   = norm(registered)
    reg_arr[sitk.GetArrayFromImage(registered) < 0] = 0   # mask sentinel
    mask_arr  = (sitk.GetArrayFromImage(mask) > 0.5).astype(np.float32)

    return fixed_arr, reg_arr, mask_arr


def make_rgb(fixed_sl, moving_sl, mask_sl):
    """Dual-colour overlay: fixed=red, moving=cyan, overlap=grey."""
    rgb = np.zeros((*fixed_sl.shape, 3), dtype=np.float32)
    rgb[..., 0] = fixed_sl                  # red channel  = fixed
    rgb[..., 1] = moving_sl * mask_sl       # green channel = moving
    rgb[..., 2] = moving_sl * mask_sl       # blue channel  = moving
    return np.clip(rgb, 0, 1)

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

stack_options = list(ALL_STACKS)

fixed_dd = Dropdown(
    options=stack_options, value=stack_options[0],
    description='Fixed:',
    layout=widgets.Layout(width='480px'),
)
moving_dd = Dropdown(
    options=stack_options,
    value=stack_options[min(1, len(stack_options)-1)],
    description='Moving:',
    layout=widgets.Layout(width='480px'),
)
transform_dd = Dropdown(
    options=['affine', 'rigid'], value='affine',
    description='Transform:',
    layout=widgets.Layout(width='200px'),
)
register_btn = Button(
    description='Register',
    button_style='primary',
    layout=widgets.Layout(width='120px'),
)
slice_sl = IntSlider(
    min=0, max=1, step=1, value=0,
    description='Slice:',
    layout=widgets.Layout(width='600px'),
)
status_lbl = HTML(value='<i>Select images and click Register.</i>')
out = widgets.Output()

# State
_result = {'fixed': None, 'registered': None, 'mask': None}


def render_slice(slice_idx):
    if _result['fixed'] is None:
        return
    fixed_sl = _result['fixed'][slice_idx]
    reg_sl   = _result['registered'][slice_idx]
    mask_sl  = _result['mask'][slice_idx]
    rgb      = make_rgb(fixed_sl, reg_sl, mask_sl)

    n_overlap = int(mask_sl.sum())
    n_total   = mask_sl.size
    pct       = 100 * n_overlap / n_total

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(fixed_sl, cmap='gray', origin='lower')
    axes[0].set_title(f'Fixed\n{fixed_dd.value.split(" / ")[1]}', fontsize=9)

    axes[1].imshow(_result['registered'][slice_idx], cmap='gray', origin='lower')
    axes[1].set_title(f'Registered moving\n{moving_dd.value.split(" / ")[1]}', fontsize=9)

    axes[2].imshow(rgb, origin='lower')
    axes[2].set_title(
        f'Overlap  ({pct:.0f}% covered)\n'
        'red = fixed only  |  cyan = moving only  |  grey = overlap',
        fontsize=9,
    )

    for ax in axes:
        ax.axis('off')

    fig.suptitle(f'Slice {slice_idx}', fontsize=11)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def on_register(_):
    status_lbl.value = '<b>Registering… this may take ~30 s.</b>'
    register_btn.disabled = True
    try:
        fixed_path  = ALL_STACKS[fixed_dd.value]
        moving_path = ALL_STACKS[moving_dd.value]
        f, r, m = register(fixed_path, moving_path, transform=transform_dd.value)
        _result['fixed']      = f
        _result['registered'] = r
        _result['mask']       = m
        slice_sl.max   = f.shape[0] - 1
        slice_sl.value = f.shape[0] // 2
        status_lbl.value = (
            f'<b style="color:green">Registration complete.</b>  '
            f'Fixed shape: {f.shape}'
        )
        render_slice(slice_sl.value)
    except Exception as e:
        status_lbl.value = f'<b style="color:red">Error: {e}</b>'
    finally:
        register_btn.disabled = False


register_btn.on_click(on_register)
slice_sl.observe(lambda c: render_slice(c['new']), names='value')

display(VBox([
    HBox([fixed_dd, moving_dd]),
    HBox([transform_dd, register_btn]),
    status_lbl,
    slice_sl,
    out,
]))

---\n## Fat image registration

In [ ]:
# ── Discover fat stacks ───────────────────────────────────────────────────────

def discover_huashanmyo_fat(root):
    stacks = {}
    for f in sorted((root / 'Fat').glob('THIGH_*_0000.nii.gz')):
        stem = f.name.replace('_0000.nii.gz', '')
        stacks[f'HuashanMyo / {stem}'] = f
    return stacks


def discover_myosegmentum_fat(root):
    stacks = {}
    for f in sorted(root.glob('*/ImageData/*_FAT/*_FAT_stack*.nii')):
        m = re.match(r'(.+?)_FAT_stack(\d+)\.nii$', f.name)
        if m:
            label = f'MyosegmenTUM / {m.group(1)}_stack{m.group(2)}'
            stacks[label] = f
    return stacks


ALL_FAT_STACKS = {}
if hm_root.exists():
    ALL_FAT_STACKS.update(discover_huashanmyo_fat(hm_root))
if myo_root.exists():
    ALL_FAT_STACKS.update(discover_myosegmentum_fat(myo_root))

print(f'{len(ALL_FAT_STACKS)} fat stacks available')
for k in list(ALL_FAT_STACKS)[:4]:
    print(f'  {k}')

In [ ]:
fat_options = list(ALL_FAT_STACKS)

fat_fixed_dd = Dropdown(
    options=fat_options, value=fat_options[0],
    description='Fixed:',
    layout=widgets.Layout(width='480px'),
)
fat_moving_dd = Dropdown(
    options=fat_options,
    value=fat_options[min(1, len(fat_options)-1)],
    description='Moving:',
    layout=widgets.Layout(width='480px'),
)
fat_transform_dd = Dropdown(
    options=['affine', 'rigid'], value='affine',
    description='Transform:',
    layout=widgets.Layout(width='200px'),
)
fat_register_btn = Button(
    description='Register',
    button_style='primary',
    layout=widgets.Layout(width='120px'),
)
fat_slice_sl = IntSlider(
    min=0, max=1, step=1, value=0,
    description='Slice:',
    layout=widgets.Layout(width='600px'),
)
fat_status_lbl = HTML(value='<i>Select fat images and click Register.</i>')
fat_out = widgets.Output()

_fat_result = {'fixed': None, 'registered': None, 'mask': None}


def render_fat_slice(slice_idx):
    if _fat_result['fixed'] is None:
        return
    fixed_sl = _fat_result['fixed'][slice_idx]
    reg_sl   = _fat_result['registered'][slice_idx]
    mask_sl  = _fat_result['mask'][slice_idx]
    rgb      = make_rgb(fixed_sl, reg_sl, mask_sl)

    pct = 100 * int(mask_sl.sum()) / mask_sl.size

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(fixed_sl, cmap='gray', origin='lower')
    axes[0].set_title(f'Fixed (fat)\n{fat_fixed_dd.value.split(" / ")[1]}', fontsize=9)
    axes[1].imshow(reg_sl, cmap='gray', origin='lower')
    axes[1].set_title(f'Registered moving (fat)\n{fat_moving_dd.value.split(" / ")[1]}', fontsize=9)
    axes[2].imshow(rgb, origin='lower')
    axes[2].set_title(
        f'Overlap  ({pct:.0f}% covered)\n'
        'red = fixed only  |  cyan = moving only  |  grey = overlap',
        fontsize=9,
    )
    for ax in axes:
        ax.axis('off')
    fig.suptitle(f'Fat — Slice {slice_idx}', fontsize=11)
    plt.tight_layout()
    with fat_out:
        fat_out.clear_output(wait=True)
        plt.show()


def on_fat_register(_):
    fat_status_lbl.value = '<b>Registering fat images… this may take ~30 s.</b>'
    fat_register_btn.disabled = True
    try:
        f, r, m = register(
            ALL_FAT_STACKS[fat_fixed_dd.value],
            ALL_FAT_STACKS[fat_moving_dd.value],
            transform=fat_transform_dd.value,
        )
        _fat_result['fixed']      = f
        _fat_result['registered'] = r
        _fat_result['mask']       = m
        fat_slice_sl.max   = f.shape[0] - 1
        fat_slice_sl.value = f.shape[0] // 2
        fat_status_lbl.value = (
            f'<b style="color:green">Registration complete.</b>  '
            f'Fixed shape: {f.shape}'
        )
        render_fat_slice(fat_slice_sl.value)
    except Exception as e:
        fat_status_lbl.value = f'<b style="color:red">Error: {e}</b>'
    finally:
        fat_register_btn.disabled = False


fat_register_btn.on_click(on_fat_register)
fat_slice_sl.observe(lambda c: render_fat_slice(c['new']), names='value')

display(VBox([
    HBox([fat_fixed_dd, fat_moving_dd]),
    HBox([fat_transform_dd, fat_register_btn]),
    fat_status_lbl,
    fat_slice_sl,
    fat_out,
]))